# Итоговый проект: AI-решение для автоматизации квалификации и маршрутизации B2B-лидов («AutoLead AI»)

**Выполнил:** Григорьев Пётр  
**Группа:** J4121  
**Дисциплина:** Культура ИИ: квалифицированный заказчик (Университет ИТМО, 2026)  

---

## 1. Постановка проблемы и архитектура решения
- **Контекст:** B2B SaaS и корпоративные IT-сервисы, входящий поток 1 000 заявок в день.
- **Проблема As-Is:** Ручная обработка штатом из 23 SDR-операторов занимает от 40 минут до 4 часов, себестоимость разбора 1 лида составляет 86.48 руб., бизнес теряет до 30% конверсии из-за задержек (Lead Decay).
- **Решение To-Be:** Интеллектуальный AI-агент на LangChain 1.x и LangGraph, способный в реальном времени анализировать естественный язык, обогащать данные компании через внешние реестры, скорить лидов, создавать карточки в CRM и мгновенно эскалировать VIP-сделки руководству.
- **GitHub репозиторий проекта:** https://github.com/Altorazero/itmo-ai-culture-final-project


## 2. Сбор и разведочный анализ данных (EDA)

In [1]:
import pandas as pd
import numpy as np

df_raw = pd.read_csv('raw_leads_dataset.csv')
print('Размер сырого датасета:', df_raw.shape)
print('Количество дубликатов по lead_id:', df_raw.duplicated(subset=['lead_id']).sum())
print('Пропуски в данных:\n', df_raw.isna().sum())

Размер сырого датасета: (2530, 8)
Количество дубликатов по lead_id: 30
Пропуски в данных:
lead_id           0
company_name      0
industry        126
company_size      0
contact_role      0
urgency           0
budget_rub      303
inquiry_text      0
dtype: int64


## 3. Очистка и подготовка данных для ИИ

In [2]:
import re
from sklearn.preprocessing import StandardScaler

# Дедупликация
df = df_raw.drop_duplicates(subset=['lead_id']).copy()

# Обработка пропусков
df['industry'] = df['industry'].fillna('Не указано')
median_by_size = df.groupby('company_size')['budget_rub'].transform('median')
df['budget_imputed_flag'] = df['budget_rub'].isna().astype(int)
df['budget_rub'] = df['budget_rub'].fillna(median_by_size).fillna(df['budget_rub'].median())

# Очистка текста
df['clean_inquiry'] = df['inquiry_text'].apply(lambda x: re.sub(r'\s+', ' ', re.sub(r'<[^>]+>', ' ', str(x))).strip())
df['inquiry_length'] = df['clean_inquiry'].apply(len)
df['has_phone'] = df['clean_inquiry'].str.contains(r'\+7|\b8\s?\(?\d{3}\)?', regex=True).astype(int)

# Логарифмирование и нормализация скоса
skew_before = df['budget_rub'].skew()
df['log_budget'] = np.log1p(df['budget_rub'])
skew_after = df['log_budget'].skew()

print('Уникальных записей после очистки:', len(df))
print(f'Асимметрия бюджета ДО: {skew_before:.2f}')
print(f'Асимметрия бюджета ПОСЛЕ log1p: {skew_after:.2f}')

df_proc = pd.read_csv('processed_leads_dataset.csv')
print('Распределение категорий маршрутизации:\n', df_proc['routing_category'].value_counts())

Уникальных записей после очистки: 2500
Асимметрия бюджета ДО: 6.49
Асимметрия бюджета ПОСЛЕ log1p: 0.05
Распределение категорий маршрутизации:
routing_category
Priority (Старший менеджер)     1360
Standard (Авто-квалификация)    1016
Enterprise VIP (Эскалация)       124
Name: count, dtype: int64


## 4. Прототип AI-агента (Python + LangChain 1.x / LangGraph)

In [3]:
# Исполнение прототипа агента (демонстрация логов)
import agent_prototype

=== СЦЕНАРИЙ 1: Стандартный Mid-Market лид ===
ВЫЗОВ:     enrich_company_data({'company_name': 'Компания_Вектор_145'})
РЕЗУЛЬТАТ: {"inn": "7801948210", "employees": 85, "revenue_rub": 180000000, "industry": "IT & SaaS"}
ВЫЗОВ:     crm_register_lead({'company_name': 'Компания_Вектор_145', 'score': 68, 'category': 'Priority', 'deal_size_est': 150000.0, 'summary': 'AI-квалификация лидов, интеграция с Битрикс24, штат 80 чел.', 'rep_assigned': 'Алексей Смирнов (Senior Sales Rep)'})
РЕЗУЛЬТАТ: Сделка сохранена в CRM. ID: CRM-78214, Ответственный: Алексей Смирнов (Senior Sales Rep)
ОТВЕТ:     Заявка успешно квалифицирована и зарегистрирована в CRM (ID: CRM-78214). Назначен старший менеджер Алексей Смирнов.

=== СЦЕНАРИЙ 2: Enterprise VIP Лид с эскалацией ===
ВЫЗОВ:     enrich_company_data({'company_name': 'Компания_Орион_512'})
РЕЗУЛЬТАТ: {"inn": "7704128954", "employees": 1250, "revenue_rub": 3400000000, "industry": "Производство & Ритейл"}
ВЫЗОВ:     crm_register_lead({'company_name': 'Комп

## 5. Экономическая модель и окупаемость (Unit Economics)

In [4]:
# Расчет по формуле задания
in_tok, out_tok = 2900, 440
p_in, p_out = 0.15, 0.60 # за 1M токенов
cost_api_usd = ((in_tok * p_in) + (out_tok * p_out)) / 1000000
cost_api_rub = cost_api_usd * 95.0
total_ai_unit_rub = cost_api_rub + (0.0005 * 95.0)

manual_unit_rub = (80000 * 1.302 + 10000) / 1320
delta_rub = manual_unit_rub - total_ai_unit_rub
bep_capex = 650000 / delta_rub

print(f'Стоимость вызова API: ${cost_api_usd:.6f} ({cost_api_rub:.4f} RUB)')
print(f'Полная себестоимость обработки 1 лида AI: {total_ai_unit_rub:.4f} RUB')
print(f'Себестоимость ручной обработки 1 лида SDR: {manual_unit_rub:.2f} RUB')
print(f'Экономия на одном обращении: {delta_rub:.2f} RUB (в 758 раз дешевле)')
print(f'Операционная точка безубыточности (Fixed OpEx): {int(100000 / delta_rub)} лидов/мес.')
print(f'Полная окупаемость CapEx (650 000 руб.): {int(bep_capex)} лидов (~{bep_capex/1000:.1f} дней работы)')
annual_savings = (30000 * delta_rub * 12) - (100000 * 12) - 650000
print(f'Чистый годовой экономический эффект: {annual_savings:,.0f} RUB')
print(f'ROI первого года: {(annual_savings / 650000) * 100:.1f}%')

Стоимость вызова API: $0.000699 (0.0664 RUB)
Полная себестоимость обработки 1 лида AI: 0.1139 RUB
Себестоимость ручной обработки 1 лида SDR: 86.48 RUB
Экономия на одном обращении: 86.37 RUB (в 758 раз дешевле)
Операционная точка безубыточности (Fixed OpEx): 1158 лидов/мес.
Полная окупаемость CapEx (650 000 руб.): 7526 лидов (~7.5 дней работы)
Чистый годовой экономический эффект: 29 243 540 RUB
ROI первого года: 4499.0%


## 6. Декларация использования ИИ
- **Сам:** Концепция и постановка бизнес-задачи B2B LeadGen, формулировка признаков, проектирование инструментов агента с граничными условиями docstring, системный промпт, экономический расчет и аналитические выводы.
- **С ИИ:** Генерация вспомогательного Python-кода создания выборки данных, верстка таблиц и графиков, структурирование итогового отчета.
- **Модели:** Gemini 3.8 Flash (High), GPT-4o-mini.
- **Ссылка на репозиторий с кодом, данными и отчетом:** https://github.com/Altorazero/itmo-ai-culture-final-project
